In [0]:
%sql
select * from workspace.silver.silver_airports

airport_id,airport_name,city,country,modifiedDate
A001,Gregoryland International Airport,Kristenmouth,South Georgia and the South Sandwich Islands,2026-03-11T07:36:31.235Z
A002,East Kristin International Airport,North Michaelview,Bosnia and Herzegovina,2026-03-11T07:36:31.235Z
A003,Brownland International Airport,Samuelville,Costa Rica,2026-03-11T07:36:31.235Z
A004,Meghanton International Airport,Andrewsmouth,Macedonia,2026-03-11T07:36:31.235Z
A005,East Aaron International Airport,Davishaven,Monaco,2026-03-11T07:36:31.235Z
A006,Michaelburgh International Airport,East Blake,Iceland,2026-03-11T07:36:31.235Z
A007,West Jennifer International Airport,Jillianstad,Libyan Arab Jamahiriya,2026-03-11T07:36:31.235Z
A008,Port Craig International Airport,New Lisa,French Southern Territories,2026-03-11T07:36:31.235Z
A009,New Joshuafurt International Airport,Port Jamiehaven,Pitcairn Islands,2026-03-11T07:36:31.235Z
A010,Thompsontown International Airport,Murraychester,Ireland,2026-03-11T07:36:31.235Z


**Parameters**

**Fetching Parameter & creating variables**

In [0]:
catalog = "workspace"

#Key col list
keycols= "['booking_id']"
key_col_list = eval(keycols)

#CDC col
cdc_col="modifiedDate"

#Backdated refresh
backdated_refresh=""

#Source object
source_object="silver_booking"

#Source schema
source_schema="silver"

#Target object
target_object="DimBookings"

#Target schema
target_schema="gold"

#Surrogate Key 
surrogate_key="DimBookingsKey"


**Incremenatal Data Ingestion **

In [0]:
# No Backdated Refresh
if len(backdated_refresh)==0:
  if  spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
      last_load = spark.sql(f"SELECT max({cdccol}) as last_load FROM workspace.{target_schema}.{target_object}").collect()[0][0]
  else:
      last_load = "1900-01-01 00:00:00"

else:
    last_load  = backdated_refresh

#Test the last load 
print(last_load)

1900-01-01 00:00:00


In [0]:
df_src = spark.sql(f"SELECT * FROM workspace.{source_schema}.{source_object} WHERE {cdc_col} >= '{last_load}'")

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"): 

  # Key Columns String For Incremental
  key_cols_string_incremental = ", ".join(key_col_list)

  df_trg = spark.sql(f"""SELECT {key_cols_string_incremental}, {surrogate_key}, create_date, update_date 
                      FROM {catalog}.{target_schema}.{target_object}""")


else:

  # Key Columns String For Initial
  key_cols_string_init = [f"'' AS {i}" for i in key_col_list]
  key_cols_string_init = ", ".join(key_cols_string_init)
  
  df_trg = spark.sql(f"""SELECT {key_cols_string_init}, CAST('0' AS INT) AS {surrogate_key}, CAST('1900-01-01 00:00:00' AS timestamp) AS          create_date, CAST('1900-01-01 00:00:00' AS timestamp) AS update_date WHERE 1=0""")

In [0]:
df_trg.display()

booking_id,DimBookingsKey,create_date,update_date


In [0]:
join_condition = ' AND '.join([f"src.{i} = trg.{i}" for i in key_col_list])

In [0]:
df_src.createOrReplaceTempView("src")
df_trg.createOrReplaceTempView("trg")

df_join = spark.sql(f"""
            SELECT src.*, 
                   trg.{surrogate_key},
                   trg.create_date,
                   trg.update_date
            FROM src
            LEFT JOIN trg
            ON {join_condition}
            """)

In [0]:
df_join.display()

booking_id,passenger_id,flight_id,airport_id,amount,booking_date,modifiedDate,DimBookingsKey,create_date,update_date
B00001,P0048,F0014,A048,850.72,2025-05-29,2026-03-11T07:36:39.845Z,null,null,null
B00002,P0011,F0052,A003,376.63,2025-06-09,2026-03-11T07:36:39.845Z,null,null,null
B00003,P0079,F0023,A012,534.02,2025-06-03,2026-03-11T07:36:39.845Z,null,null,null
B00004,P0068,F0001,A039,1333.7,2025-06-16,2026-03-11T07:36:39.845Z,null,null,null
B00005,P0189,F0019,A008,1334.96,2025-06-17,2026-03-11T07:36:39.845Z,null,null,null
B00006,P0070,F0003,A004,296.13,2025-05-18,2026-03-11T07:36:39.845Z,null,null,null
B00007,P0194,F0068,A016,460.14,2025-04-05,2026-03-11T07:36:39.845Z,null,null,null
B00008,P0181,F0048,A017,1402.02,2025-06-04,2026-03-11T07:36:39.845Z,null,null,null
B00009,P0096,F0081,A016,1444.51,2025-05-16,2026-03-11T07:36:39.845Z,null,null,null
B00010,P0131,F0089,A034,292.39,2025-05-16,2026-03-11T07:36:39.845Z,null,null,null


In [0]:
from pyspark.sql.functions import *


# OLD RECORDS
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())

# NEW RECOERDS
df_new = df_join.filter(col(f'{surrogate_key}').isNull())

In [0]:
df_old_enr = df_old.withColumn('update_date', current_timestamp())

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"): 
    max_surrogate_key = spark.sql(f"""
                            SELECT max({surrogate_key}) FROM {catalog}.{target_schema}.{target_object}
                        """).collect()[0][0]
    df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
                    .withColumn('create_date', current_timestamp())\
                    .withColumn('update_date', current_timestamp())    

else:
    max_surrogate_key = 0
    df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
                    .withColumn('create_date', current_timestamp())\
                    .withColumn('update_date', current_timestamp())


In [0]:
df_union = df_old_enr.unionByName(df_new_enr)

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_union.alias("src"), f"trg.{surrogate_key} = src.{surrogate_key}")\
                        .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
                        .whenNotMatchedInsertAll()\
                        .execute()

else: 

    df_union.write.format("delta")\
            .mode("append")\
            .saveAsTable(f"{catalog}.{target_schema}.{target_object}")


In [0]:
%sql
select * from workspace.gold.dimbookings

booking_id,passenger_id,flight_id,airport_id,amount,booking_date,modifiedDate,DimBookingsKey,create_date,update_date
B00001,P0048,F0014,A048,850.72,2025-05-29,2026-03-11T07:36:39.845Z,1,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
B00002,P0011,F0052,A003,376.63,2025-06-09,2026-03-11T07:36:39.845Z,2,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
B00003,P0079,F0023,A012,534.02,2025-06-03,2026-03-11T07:36:39.845Z,3,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
B00004,P0068,F0001,A039,1333.7,2025-06-16,2026-03-11T07:36:39.845Z,4,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
B00005,P0189,F0019,A008,1334.96,2025-06-17,2026-03-11T07:36:39.845Z,5,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
B00006,P0070,F0003,A004,296.13,2025-05-18,2026-03-11T07:36:39.845Z,6,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
B00007,P0194,F0068,A016,460.14,2025-04-05,2026-03-11T07:36:39.845Z,7,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
B00008,P0181,F0048,A017,1402.02,2025-06-04,2026-03-11T07:36:39.845Z,8,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
B00009,P0096,F0081,A016,1444.51,2025-05-16,2026-03-11T07:36:39.845Z,9,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
B00010,P0131,F0089,A034,292.39,2025-05-16,2026-03-11T07:36:39.845Z,10,2026-03-17T09:30:53.465Z,2026-03-17T09:30:53.465Z
